In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

In [1]:
FRAME = "000000"

VELO_PATH = f"../data/KITTI/training/velodyne/{FRAME}.bin"
CALIB_PATH = f"../data/KITTI/training/calib/{FRAME}.txt"
LABEL_PATH = f"../data/KITTI/training/label_2/{FRAME}.txt"

xyz, intensity = _3d_lidar_points(VELO_PATH)

tr_velo_to_cam, r0_rect, p2 = transformation_matrices(CALIB_PATH)

objects = read_labels(LABEL_PATH)

print("LiDAR points:", xyz.shape)
print("Number of labels:", len(objects))

for obj in objects:
    print(obj["type"], obj["dimensions"], obj["location"])

NameError: name '_3d_lidar_points' is not defined

In [ ]:
car = next(obj for obj in objects if obj["type"] == "Car")

print("Type:")
print(car["type"])

print("\nDimensions [h, w, l]:")
print(car["dimensions"])

print("\nLocation [x, y, z] in camera coordinates:")
print(car["location"])

print("\nrotation_y:")
print(car["rotation_y"])

In [ ]:
def create_3d_box_camera(dimensions, location, rotation_y):
    """
    Create the 8 corners of a KITTI 3D bounding box
    in the rectified camera coordinate system.

    dimensions = [height, width, length]
    location   = [x, y, z]  (bottom center)
    rotation_y = rotation around camera Y axis
    """

    h, w, l = dimensions
    x, y, z = location

    # Corners before rotation
    #
    # x -> left/right along object length
    # y -> vertical
    # z -> front/back along object width
    #
    # Bottom four corners
    # Top four corners

    corners = np.array([
        [ l/2,  0,  w/2],
        [ l/2,  0, -w/2],
        [-l/2,  0, -w/2],
        [-l/2,  0,  w/2],

        [ l/2, -h,  w/2],
        [ l/2, -h, -w/2],
        [-l/2, -h, -w/2],
        [-l/2, -h,  w/2],
    ])

    # Rotation around camera Y axis
    c = np.cos(rotation_y)
    s = np.sin(rotation_y)

    R = np.array([
        [ c, 0, s],
        [ 0, 1, 0],
        [-s, 0, c]
    ])

    # Rotate
    corners = (R @ corners.T).T

    # Translate
    corners += np.array([x, y, z])

    return corners

In [ ]:
box_camera = create_3d_box_camera(
    car["dimensions"],
    car["location"],
    car["rotation_y"]
)

print(box_camera)
print("Shape:", box_camera.shape)

In [ ]:
fig = plt.figure(figsize=(8, 7))
ax = fig.add_subplot(111, projection="3d")

ax.scatter(
    box_camera[:, 0],
    box_camera[:, 2],
    -box_camera[:, 1],
    s=50
)

ax.set_xlabel("Camera X")
ax.set_ylabel("Camera Z")
ax.set_zlabel("-Camera Y")

ax.set_title("KITTI 3D Bounding Box — Camera Coordinates")

plt.show()

In [ ]:
BOX_EDGES = [
    (0, 1),
    (1, 2),
    (2, 3),
    (3, 0),

    (4, 5),
    (5, 6),
    (6, 7),
    (7, 4),

    (0, 4),
    (1, 5),
    (2, 6),
    (3, 7),
]

In [ ]:
fig = plt.figure(figsize=(8, 7))
ax = fig.add_subplot(111, projection="3d")

ax.scatter(
    box_camera[:, 0],
    box_camera[:, 2],
    -box_camera[:, 1],
    s=50
)

for i, j in BOX_EDGES:
    ax.plot(
        [box_camera[i, 0], box_camera[j, 0]],
        [box_camera[i, 2], box_camera[j, 2]],
        [-box_camera[i, 1], -box_camera[j, 1]]
    )

ax.set_xlabel("Camera X")
ax.set_ylabel("Camera Z")
ax.set_zlabel("-Camera Y")

ax.set_title(f"3D {car['type']} Bounding Box — Camera Coordinates")

plt.show()

In [ ]:
def camera_to_lidar(points_camera, tr_velo_to_cam, r0_rect):
    """
    Transform points from rectified camera coordinates
    back into LiDAR coordinates.
    """

    # Inverse rectification
    points_camera_unrect = (
        np.linalg.inv(r0_rect) @ points_camera.T
    ).T

    # Convert 3x4 transformation into 4x4
    T = np.eye(4)
    T[:3, :] = tr_velo_to_cam

    T_inv = np.linalg.inv(T)

    # Homogeneous coordinates
    points_h = np.hstack([
        points_camera_unrect,
        np.ones((points_camera_unrect.shape[0], 1))
    ])

    points_lidar = (T_inv @ points_h.T).T

    return points_lidar[:, :3]

In [ ]:
box_lidar = camera_to_lidar(
    box_camera,
    tr_velo_to_cam,
    r0_rect
)

print(box_lidar)

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")

# LiDAR points
ax.scatter(
    xyz[:, 0],
    xyz[:, 1],
    xyz[:, 2],
    s=0.5,
    alpha=0.3
)

# 3D bounding box
for i, j in BOX_EDGES:
    ax.plot(
        [box_lidar[i, 0], box_lidar[j, 0]],
        [box_lidar[i, 1], box_lidar[j, 1]],
        [box_lidar[i, 2], box_lidar[j, 2]],
        linewidth=2
    )

ax.set_xlabel("LiDAR X — Forward")
ax.set_ylabel("LiDAR Y — Left")
ax.set_zlabel("LiDAR Z — Up")

ax.set_title(
    f"LiDAR Point Cloud + Ground Truth 3D {car['type']}"
)

plt.show()

In [ ]:
def lidar_to_rectified_camera(xyz, tr_velo_to_cam, r0_rect):

    xyz_h = np.hstack([
        xyz,
        np.ones((xyz.shape[0], 1))
    ])

    camera_xyz = (tr_velo_to_cam @ xyz_h.T).T

    rectified_xyz = (r0_rect @ camera_xyz.T).T

    return rectified_xyz

In [ ]:
rectified_points = lidar_to_rectified_camera(
    xyz,
    tr_velo_to_cam,
    r0_rect
)

In [ ]:
def points_in_3d_box(points_camera, dimensions, location, rotation_y):

    h, w, l = dimensions

    # Move origin to object location
    points = points_camera - location

    # Undo object rotation
    c = np.cos(rotation_y)
    s = np.sin(rotation_y)

    R = np.array([
        [ c, 0, -s],
        [ 0, 1,  0],
        [ s, 0,  c]
    ])

    local_points = (R @ points.T).T

    # KITTI box coordinates:
    #
    # x: [-l/2, l/2]
    # y: [-h, 0]
    # z: [-w/2, w/2]

    mask = (
        (local_points[:, 0] >= -l/2) &
        (local_points[:, 0] <=  l/2) &
        (local_points[:, 1] >= -h) &
        (local_points[:, 1] <=  0) &
        (local_points[:, 2] >= -w/2) &
        (local_points[:, 2] <=  w/2)
    )

    return mask

In [ ]:
mask = points_in_3d_box(
    rectified_points,
    car["dimensions"],
    car["location"],
    car["rotation_y"]
)

car_points = xyz[mask]

print("Total LiDAR points:", len(xyz))
print("Points inside car box:", len(car_points))

In [ ]:
fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection="3d")

ax.scatter(
    car_points[:, 0],
    car_points[:, 1],
    car_points[:, 2],
    s=5
)

ax.set_xlabel("LiDAR X — Forward")
ax.set_ylabel("LiDAR Y — Left")
ax.set_zlabel("LiDAR Z — Up")

ax.set_title(
    f"LiDAR Points Inside Ground Truth {car['type']} Box"
)

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

# All LiDAR points
ax.scatter(
    xyz[:, 0],
    xyz[:, 1],
    s=0.3,
    alpha=0.15
)

# Object points
ax.scatter(
    car_points[:, 0],
    car_points[:, 1],
    s=5
)

# 3D box in BEV
for i, j in BOX_EDGES:

    # Only draw edges belonging to the bottom/top rectangles
    if i < 4 and j < 4:
        ax.plot(
            [box_lidar[i, 0], box_lidar[j, 0]],
            [box_lidar[i, 1], box_lidar[j, 1]],
            linewidth=2
        )

ax.set_xlabel("LiDAR X — Forward (m)")
ax.set_ylabel("LiDAR Y — Left (m)")
ax.set_title("BEV — LiDAR + Ground Truth Car")

ax.set_aspect("equal")

plt.show()